In [9]:
import os
import pandas as pd
from pathlib import Path

DATASET_ROOT = "/kaggle/input/datasets/pevogam/ucf101/UCF101/UCF-101"

SPLIT_ROOT = (
    "/kaggle/input/datasets/pevogam/ucf101/"
    "UCF101TrainTestSplits-RecognitionTask/ucfTrainTestlist"
)

# Load class mapping
class_file = os.path.join(SPLIT_ROOT, "classInd.txt")

class_rows = []
with open(class_file, "r") as f:
    for line in f:
        class_id, class_name = line.strip().split()
        class_rows.append((int(class_id), class_name))

class_df = pd.DataFrame(
    class_rows, columns=["label", "class_name"]
)

class_to_idx = {
    name: label - 1
    for label, name in class_rows
}

# Load official training split 1
train_rows = []

with open(os.path.join(SPLIT_ROOT, "trainlist01.txt"), "r") as f:
    for line in f:
        relative_path = line.strip().split()[0]
        class_name = relative_path.split("/")[0]

        train_rows.append({
            "video_path": os.path.join(DATASET_ROOT, relative_path),
            "class_name": class_name,
            "label_idx": class_to_idx[class_name],
            "split": "train"
        })

# Load official test split 1
test_rows = []

with open(os.path.join(SPLIT_ROOT, "testlist01.txt"), "r") as f:
    for line in f:
        relative_path = line.strip()
        class_name = relative_path.split("/")[0]

        test_rows.append({
            "video_path": os.path.join(DATASET_ROOT, relative_path),
            "class_name": class_name,
            "label_idx": class_to_idx[class_name],
            "split": "test"
        })

df = pd.DataFrame(train_rows + test_rows)

print("Manifest rebuilt.")
print("Total videos:", len(df))
print("Official training:", (df["split"] == "train").sum())
print("Official testing:", (df["split"] == "test").sum())
print("Classes:", df["class_name"].nunique())



Manifest rebuilt.
Total videos: 13320
Official training: 9537
Official testing: 3783
Classes: 101


In [10]:
from sklearn.model_selection import train_test_split

# Separate official training and testing data
official_train = df[df["split"] == "train"].copy()
official_test = df[df["split"] == "test"].copy()

# Extract source group from video filename
official_train["source_group"] = (
    official_train["video_path"]
    .str.extract(r"_(g\d+)_")[0]
)

official_test["source_group"] = (
    official_test["video_path"]
    .str.extract(r"_(g\d+)_")[0]
)

train_parts = []
val_parts = []

# Split by source group within each class
for class_name, class_data in official_train.groupby("class_name"):

    groups = sorted(class_data["source_group"].unique())

    train_groups, val_groups = train_test_split(
        groups,
        test_size=0.15,
        random_state=42
    )

    train_parts.append(
        class_data[class_data["source_group"].isin(train_groups)]
    )

    val_parts.append(
        class_data[class_data["source_group"].isin(val_groups)]
    )

train_df = pd.concat(train_parts, ignore_index=True)
val_df = pd.concat(val_parts, ignore_index=True)

train_df["split"] = "train"
val_df["split"] = "validation"
official_test["split"] = "test"

df = pd.concat(
    [train_df, val_df, official_test],
    ignore_index=True
)

print("RESTORED DATASET SPLIT")
print("----------------------")
print(df["split"].value_counts())
print("Total:", len(df))
print("Classes:", df["class_name"].nunique())

# Verify no source-group overlap within each class
for class_name in df["class_name"].unique():

    class_data = df[df["class_name"] == class_name]

    train_groups = set(
        class_data[class_data["split"] == "train"]["source_group"]
    )

    val_groups = set(
        class_data[class_data["split"] == "validation"]["source_group"]
    )

    assert train_groups.isdisjoint(val_groups)

print("Source-group leakage check: PASSED")

RESTORED DATASET SPLIT
----------------------
split
train         7933
test          3783
validation    1604
Name: count, dtype: int64
Total: 13320
Classes: 101
Source-group leakage check: PASSED


## Dataset Partitioning and Leakage Prevention

The UCF101 dataset was partitioned using the official Split 1 training and testing lists.

The official training data was further divided into training and validation subsets using a group-based stratified procedure within each action class.

- **Training:** 7,933 videos
- **Validation:** 1,604 videos
- **Testing:** 3,783 videos
- **Total:** 13,320 videos
- **Action classes:** 101

A fixed random seed (`random_state=42`) was used to ensure reproducibility. Source-group overlap between the training and validation subsets was checked and confirmed to be absent.

The official test set remains separate and will be reserved for final evaluation.

**Leakage Check:** PASSED.

### Stage 2: Dataset Integrity Audit — Completed

The complete UCF101 video integrity audit was successfully repeated.

- Total videos audited: 13,320
- Fully readable videos: 13,096
- Partially readable videos: 224
- Unreadable videos: 0
- Audit duration: 772.52 seconds

All videos contain at least one successfully decoded frame. Partially readable videos are retained using their actual decoded frame counts.

**Output:** `full_video_integrity_audit.csv`

**Next:** Restore the audited manifest, video Dataset class, and DataLoaders.

### Project Recovery — November 22, 2026

The Kaggle session restarted and the previous training checkpoint was not preserved.

**Previously completed work:**
- UCF101 dataset preparation and integrity audit.
- 32-frame candidate extraction pipeline.
- Uniform and temporal-difference sampling methods.
- MobileNetV3-Large video classifier.
- First baseline training epoch.

**Previously recorded baseline results:**
- Training videos: 7,933
- Frame budget: 4
- Training loss: 2.7532
- Training accuracy: 34.35%
- Training time: 901.24 seconds

**Recovery plan:** Restore the dataset pipeline, retrain the baseline, and immediately download the model checkpoint to local storage.

### Stage 1: Dataset Preparation and Model Pipeline Validation

**Completed tasks:**
- Verified the UCF101 dataset containing 13,320 videos across 101 action classes.
- Prepared training (7,933), validation (1,604), and testing (3,783) partitions.
- Completed video integrity auditing and generated an audited dataset manifest.
- Developed and tested sequential video decoding and 32-frame candidate extraction.
- Implemented and validated two frame-selection strategies:
  - Uniform temporal sampling
  - Temporal-difference-based adaptive sampling
- Tested frame budgets of 4, 8, 12, and 16.
- Configured MobileNetV3-Large with a 101-class output layer.
- Successfully completed a 10-batch GPU training pilot.

**Initial status:** The data pipeline, sampling methods, classification model, and initial GPU training workflow are operational.

**Next stage:** Conduct controlled baseline and proposed-method experiments, followed by validation and computational-efficiency evaluation.

*Note: The pilot is a pipeline test, not a final model-performance evaluation.*